# Diagnóstico da Relação MP ↔ SKP (Matéria-Prima ↔ Produto)

## Objetivo

Diagnosticar a disponibilização de matérias-primas (MP/artigos) em relação aos produtos (SKPs), identificando:

1. **Concentração MP → SKP**: matérias-primas que atendem poucos produtos, gerando risco de baixa flexibilidade no portfólio.
2. **Divergência de cores dentro da mesma MP**: quando uma MP é compartilhada entre produtos, mas determinada cor aparece em apenas um produto — funcionando como uma "MP-cor" exclusiva e reduzindo o reaproveitamento.

## Escopo

- Foco exclusivo em diagnóstico e oportunidades. Nenhuma decisão de negócio é tomada.
- Sem análise de demanda, MOQ, margem ou lote mínimo.
- Cores com nomes diferentes são tratadas como cores diferentes (sem tabela de equivalência).

## 2. Imports e Configuração

In [37]:
from google.cloud import bigquery
import pandas as pd
import numpy as np

client = bigquery.Client(project="insider-data-lake")


def query_to_dataframe(query):
    query_job = client.query(query)
    results = query_job.result()
    return results.to_dataframe()

/Users/insider/LA_Coding_Projects/.venv/lib/python3.13/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


## 3. Carregamento da Base

Duas queries são executadas:
1. **Base principal (MP × SKP)**: grão `product_name × article_name`, com consumo mediano e status do produto. Inclui apenas produtos com vendas nos últimos 8 meses.
2. **Base de cores (MP × SKP × cor)**: grão `product_name × article_name × color`, necessária para o Diagnóstico 2.

### Filtros aplicados na query:
- Apenas produtos com ao menos 1 venda nos últimos 8 meses (L8M)
- Status: `ativo_perene`, `ativo_em_lancamento` ou `desativado`
- Excluídos: produtos Ziraldo, XP, Maluquinho, B2B

In [38]:
query_base = """
    WITH fabric_costs AS (
        SELECT
            mfs.id AS fabric_sku_id,
            mfs.fabric_id,
            mfs.knitting_factory_id,
            mfs.sku AS fabric_sku,
            mfs.invoice_fabric_name AS factory_fabric_name,
            mfs.unit_price,
            mfs.minimum_volume_per_order,
            mfs.multiple_volume_per_order,
            mf.name AS fabric_name,
            mf.article_id,
            ma.name AS article_name,
            ma.unit AS article_unit,
            mkf.supplier_id,
            ms.alias AS knitting_factory_name
        FROM `insider-data-lake.integrated.muninn_fabric_skus` AS mfs
        LEFT JOIN `insider-data-lake.integrated.muninn_fabrics` AS mf ON mf.id = mfs.fabric_id
        LEFT JOIN `insider-data-lake.integrated.muninn_articles` AS ma ON ma.id = mf.article_id
        LEFT JOIN `insider-data-lake.integrated.muninn_knitting_factories` AS mkf ON mkf.id = mfs.knitting_factory_id
        LEFT JOIN `insider-data-lake.integrated.muninn_suppliers` AS ms ON ms.id = mkf.supplier_id
    ),
    fabric_min_max_cost AS (
        SELECT
            fc.fabric_id,
            fc.fabric_name,
            MIN(fc.unit_price) AS min_fabric_cost,
            MAX(fc.unit_price) AS max_fabric_cost,
            MIN(fc.minimum_volume_per_order) AS minimum_volume_per_order,
            COUNT(DISTINCT fc.knitting_factory_id) AS number_knitting_factories,
            ARRAY_AGG(DISTINCT fc.knitting_factory_name) AS knitting_factories_names,
        FROM fabric_costs AS fc
        GROUP BY fc.fabric_id, fc.fabric_name
    ),
    skp_with_sales_l8m AS (
        SELECT DISTINCT s.product_name
        FROM `insider-data-lake.fpa.dre` d
        LEFT JOIN `insider-data-lake.integrated.skus` s USING(sku)
        WHERE DATE(d.order_date) >= DATE_SUB(CURRENT_DATE(), INTERVAL 8 MONTH)
        AND d.order_status != 'Not authorized'
        AND d.quantity > 0
        AND s.product_name IS NOT NULL
    ),
    skp_status AS (
        SELECT
            s.product_name,
            CASE
                WHEN COUNTIF(s.sku_state = 'ativo_perene') > 0         THEN 'ativo_perene'
                WHEN COUNTIF(s.sku_state = 'ativo_em_lancamento') > 0  THEN 'ativo_em_lancamento'
                WHEN COUNTIF(s.sku_state = 'ativo_capsula') > 0        THEN 'ativo_capsula'
                WHEN COUNTIF(s.sku_state = 'personalizacao') > 0       THEN 'personalizacao'
                WHEN COUNTIF(s.sku_state = 'kit') > 0                  THEN 'kit'
                ELSE 'desativado'
            END AS product_status
        FROM `insider-data-lake.integrated.skus` s
        INNER JOIN skp_with_sales_l8m l8m USING(product_name)
        GROUP BY s.product_name
    ),
    sku_fabrics AS (
        SELECT
            mps.sku,
            mps.sku_name,
            s.sku_state,
            s.gender,
            s.color,
            s.size,
            s.product_name,
            mpsf.fabric_id,
            mf.name AS fabric_name,
            mpsf.consumption,
            fc.min_fabric_cost AS min_fabric_unitary_cost,
            fc.max_fabric_cost AS max_fabric_unitary_cost,
            fc.minimum_volume_per_order,
            fc.min_fabric_cost * mpsf.consumption AS min_fabric_cost,
            fc.max_fabric_cost * mpsf.consumption AS max_fabric_cost,
            ma.unit AS article_unit,
            ma.name AS article_name,
            mf.article_id,
            fc.number_knitting_factories,
            fc.knitting_factories_names
        FROM `insider-data-lake.integrated.muninn_product_skus_fabrics` AS mpsf
        LEFT JOIN `insider-data-lake.integrated.muninn_fabrics` AS mf ON mf.id = mpsf.fabric_id
        LEFT JOIN `insider-data-lake.integrated.muninn_articles` AS ma ON ma.id = mf.article_id
        LEFT JOIN `insider-data-lake.integrated.muninn_product_skus` AS mps ON mps.product_sku_id = mpsf.product_sku_id
        LEFT JOIN `insider-data-lake.integrated.skus` AS s ON mps.sku = s.sku
        LEFT JOIN fabric_min_max_cost AS fc ON fc.fabric_id = mpsf.fabric_id
        INNER JOIN skp_with_sales_l8m l8m ON l8m.product_name = s.product_name
    )

    -- Base principal: grão product_name x article_name
    SELECT
        sf.product_name,
        ss.product_status,
        REGEXP_REPLACE(sf.article_name, r'Modal \\(\\d+\\)', 'Modal') AS article_name,
        sf.article_unit,
        MIN(sf.minimum_volume_per_order) AS minimum_volume_per_order,
        APPROX_QUANTILES(sf.consumption, 2)[OFFSET(1)] AS median_article_consumption
    FROM sku_fabrics AS sf
    INNER JOIN skp_status AS ss ON ss.product_name = sf.product_name
    WHERE ss.product_status IN ('ativo_perene', 'ativo_em_lancamento', 'desativado')
    AND LOWER(sf.product_name) NOT LIKE '%ziraldo%'
    AND LOWER(sf.product_name) NOT LIKE '% xp%'
    AND LOWER(sf.product_name) NOT LIKE '%maluquinho%'
    AND LOWER(sf.product_name) NOT LIKE '% b2b %'
    GROUP BY sf.product_name, ss.product_status, article_name, sf.article_unit
    ORDER BY sf.product_name ASC
"""

df = query_to_dataframe(query_base)
print(f"Base principal carregada: {df.shape[0]} linhas, {df.shape[1]} colunas")
df.head(10)

/Users/insider/LA_Coding_Projects/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Base principal carregada: 231 linhas, 6 colunas


,product_name,product_status,article_name,article_unit,minimum_volume_per_order,median_article_consumption
0,Action Top Feminino,desativado,Sportiva Pro,kg,153,0.150000000
1,Air Blouse 2.0 Feminino,desativado,New String Stretch,m,78,0.392000000
2,Air Blouse Feminino,desativado,String Stretch,kg,153,0.392000000
3,Air Loop Top 2.0 Feminino,desativado,New String Stretch,m,78,0.330000000
4,Air Loop Top Feminino,desativado,String Stretch,kg,153,0.330000000
5,Air Tank 2.0 Feminino,desativado,New String Stretch,m,78,0.181000000
6,Air Tank Feminino,desativado,String Stretch,kg,153,0.181000000
7,Bermuda Kyoto Feminino,ativo_perene,Nylon WR 50+,m,60,0.900000000
8,Blusa Manga Longa Comfy InLounge,desativado,Molecton Viscose,kg,<NA>,0.580000000
9,Blusa de Alça FutureForm Feminino,ativo_em_lancamento,Hydro Nylon Ultra Spandex UV50+,m,<NA>,0.190000000


In [39]:
query_cores = """
    WITH skp_with_sales_l8m AS (
        SELECT DISTINCT s.product_name
        FROM `insider-data-lake.fpa.dre` d
        LEFT JOIN `insider-data-lake.integrated.skus` s USING(sku)
        WHERE DATE(d.order_date) >= DATE_SUB(CURRENT_DATE(), INTERVAL 8 MONTH)
        AND d.order_status != 'Not authorized'
        AND d.quantity > 0
        AND s.product_name IS NOT NULL
    ),
    skp_status AS (
        SELECT
            s.product_name,
            CASE
                WHEN COUNTIF(s.sku_state = 'ativo_perene') > 0         THEN 'ativo_perene'
                WHEN COUNTIF(s.sku_state = 'ativo_em_lancamento') > 0  THEN 'ativo_em_lancamento'
                WHEN COUNTIF(s.sku_state = 'ativo_capsula') > 0        THEN 'ativo_capsula'
                WHEN COUNTIF(s.sku_state = 'ativo_personalizacao') > 0 THEN 'ativo_personalizacao'
                WHEN COUNTIF(s.sku_state = 'kit') > 0                  THEN 'kit'
                ELSE 'desativado'
            END AS product_status
        FROM `insider-data-lake.integrated.skus` s
        INNER JOIN skp_with_sales_l8m l8m USING(product_name)
        GROUP BY s.product_name
    ),
    -- Consolidar sku_state por product_name × color usando hierarquia de prioridade
    color_status_consolidated AS (
        SELECT
            s.product_name,
            s.color,
            CASE
                WHEN COUNTIF(s.sku_state = 'ativo_perene') > 0         THEN 'ativo_perene'
                WHEN COUNTIF(s.sku_state = 'ativo_em_lancamento') > 0  THEN 'ativo_em_lancamento'
                WHEN COUNTIF(s.sku_state = 'ativo_capsula') > 0        THEN 'ativo_capsula'
                WHEN COUNTIF(s.sku_state = 'ativo_personalizacao') > 0 THEN 'ativo_personalizacao'
                WHEN COUNTIF(s.sku_state = 'kit') > 0                  THEN 'kit'
                ELSE 'desativado'
            END AS color_status
        FROM `insider-data-lake.integrated.skus` s
        INNER JOIN skp_with_sales_l8m l8m USING(product_name)
        WHERE s.color IS NOT NULL
        GROUP BY s.product_name, s.color
    )

    SELECT DISTINCT
        s.product_name,
        ss.product_status,
        REGEXP_REPLACE(ma.name, r'Modal \\(\\d+\\)', 'Modal') AS article_name,
        s.color,
        csc.color_status
    FROM `insider-data-lake.integrated.muninn_product_skus_fabrics` AS mpsf
    LEFT JOIN `insider-data-lake.integrated.muninn_fabrics` AS mf ON mf.id = mpsf.fabric_id
    LEFT JOIN `insider-data-lake.integrated.muninn_articles` AS ma ON ma.id = mf.article_id
    LEFT JOIN `insider-data-lake.integrated.muninn_product_skus` AS mps ON mps.product_sku_id = mpsf.product_sku_id
    LEFT JOIN `insider-data-lake.integrated.skus` AS s ON mps.sku = s.sku
    INNER JOIN skp_with_sales_l8m l8m ON l8m.product_name = s.product_name
    INNER JOIN skp_status AS ss ON ss.product_name = s.product_name
    INNER JOIN color_status_consolidated csc
        ON csc.product_name = s.product_name AND csc.color = s.color
    WHERE ss.product_status IN ('ativo_perene', 'ativo_em_lancamento', 'ativo_capsula', 'desativado')
    AND s.color IS NOT NULL
    AND LOWER(s.product_name) NOT LIKE '%ziraldo%'
    AND LOWER(s.product_name) NOT LIKE '% xp%'
    AND LOWER(s.product_name) NOT LIKE '%maluquinho%'
    AND LOWER(s.product_name) NOT LIKE '% b2b %'
    ORDER BY article_name, product_name, color
"""

df_cores = query_to_dataframe(query_cores)
print(f"Base de cores carregada: {df_cores.shape[0]} linhas, {df_cores.shape[1]} colunas")
print()
print("Distribuição de color_status:")
print(df_cores['color_status'].value_counts())
print()
print("Distribuição de product_status:")
print(df_cores['product_status'].value_counts())
df_cores

/Users/insider/LA_Coding_Projects/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Base de cores carregada: 1488 linhas, 5 colunas

Distribuição de color_status:
color_status
desativado              1008
ativo_perene             372
ativo_em_lancamento       67
ativo_capsula             38
ativo_personalizacao       3
Name: count, dtype: int64

Distribuição de product_status:
product_status
ativo_perene           1005
desativado              387
ativo_em_lancamento      96
Name: count, dtype: int64


,product_name,product_status,article_name,color,color_status
0,Calça Flare InSkin Feminino,ativo_perene,Adapt Fit,Preto,ativo_perene
1,Modern Top Feminino,ativo_perene,Adapt Fit,Preto,ativo_perene
2,Post-Modern Top Feminino,desativado,Adapt Fit,Preto,desativado
3,Stirrup Legging Feminino,ativo_perene,Adapt Fit,Preto,ativo_perene
4,Zipper Legging Feminino,ativo_perene,Adapt Fit,Preto,ativo_perene
...,...,...,...,...,...
1483,Shorts Esportivo Endorphine Masculino,ativo_perene,Walk Stretch,Earth Brown,desativado
1484,Shorts Esportivo Endorphine Masculino,ativo_perene,Walk Stretch,Flame Red,desativado
1485,Shorts Esportivo Endorphine Masculino,ativo_perene,Walk Stretch,Marinho,desativado
1486,Shorts Esportivo Endorphine Masculino,ativo_perene,Walk Stretch,Mist Green,ativo_perene


In [40]:
# Carregar category_4 por product_name da fpa.dre
query_categorias = """
    SELECT DISTINCT
        product_name,
        category_4
    FROM `insider-data-lake.fpa.dre`
    WHERE product_name IS NOT NULL
    AND category_4 IS NOT NULL
"""

df_categorias = query_to_dataframe(query_categorias)
df_categorias['product_name'] = df_categorias['product_name'].str.strip()
df_categorias['category_4'] = df_categorias['category_4'].str.strip()

# Deduplicar: se um product_name tem múltiplas category_4, pegar a mais frequente
df_categorias = (
    df_categorias
    .groupby('product_name')['category_4']
    .agg(lambda x: x.value_counts().index[0])
    .reset_index()
)


/Users/insider/LA_Coding_Projects/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


## 4. Padronização e Filtros

In [41]:
# Padronizar nomes de colunas
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df_cores.columns = df_cores.columns.str.strip().str.lower().str.replace(' ', '_')

# Padronizar strings
for col in ['product_name', 'article_name', 'product_status']:
    df[col] = df[col].str.strip()

for col in ['product_name', 'article_name', 'product_status', 'color', 'color_status']:
    df_cores[col] = df_cores[col].str.strip()

# Checar e remover nulos em campos chave
print("Nulos na base principal:")
print(df[['product_name', 'article_name']].isnull().sum())
print()
print("Nulos na base de cores:")
print(df_cores[['product_name', 'article_name', 'color', 'color_status']].isnull().sum())

df = df.dropna(subset=['product_name', 'article_name'])
df_cores = df_cores.dropna(subset=['product_name', 'article_name', 'color'])

# Merge category_4 na base de cores
df_cores = df_cores.merge(df_categorias[['product_name', 'category_4']], on='product_name', how='left')
df_cores['category_4'] = df_cores['category_4'].fillna('Sem Categoria')

# print(f"\nBase principal após limpeza: {len(df)} linhas")
# print(f"Base de cores após limpeza: {len(df_cores)} linhas")
# print(f"\nDistribuição de category_4 na base de cores:")
# print(df_cores['category_4'].value_counts().head(10))

Nulos na base principal:
product_name    0
article_name    0
dtype: int64

Nulos na base de cores:
product_name    0
article_name    0
color           0
color_status    0
dtype: int64


## 5. Checagem do Grão da Base

In [42]:
# # Verificar unicidade do grão: product_name x article_name na base principal
# chave_principal = df.groupby(['product_name', 'article_name']).size()
# duplicadas = chave_principal[chave_principal > 1]
# print(f"Grão base principal: product_name × article_name")
# print(f"  Total de combinações: {len(chave_principal)}")
# print(f"  Combinações duplicadas: {len(duplicadas)}")
# if len(duplicadas) > 0:
#     print("  ⚠ Duplicatas encontradas — verificar antes de prosseguir:")
#     print(duplicadas.head(10))

# print()

# # Verificar unicidade do grão: product_name x article_name x color na base de cores
# chave_cores = df_cores.groupby(['product_name', 'article_name', 'color']).size()
# duplicadas_cores = chave_cores[chave_cores > 1]
# print(f"Grão base de cores: product_name × article_name × color")
# print(f"  Total de combinações: {len(chave_cores)}")
# print(f"  Combinações duplicadas: {len(duplicadas_cores)}")

# print()
# print(f"Produtos distintos (base principal): {df['product_name'].nunique()}")
# print(f"Artigos/MPs distintos (base principal): {df['article_name'].nunique()}")
# print(f"Produtos distintos (base cores): {df_cores['product_name'].nunique()}")
# print(f"Cores distintas (base cores): {df_cores['color'].nunique()}")
# print()
# print("Distribuição de product_status (base principal):")
# print(df['product_status'].value_counts())
# print()
# print("Distribuição de color_status (base cores):")
# print(df_cores['color_status'].value_counts())

## 6. Diagnóstico 1: Concentração MP → SKP

Objetivo: identificar matérias-primas que atendem poucos produtos (1 ou 2), gerando risco de baixa flexibilidade.

Para cada MP, calculamos:
- Quantidade de produtos atendidos
- Lista de produtos
- Consumo total e participação no consumo global
- Status dos produtos (priorizando perenes/ativos)

In [43]:
# Converter colunas numéricas que podem vir como Decimal do BigQuery
df['median_article_consumption'] = df['median_article_consumption'].astype(float)
df['minimum_volume_per_order'] = pd.to_numeric(df['minimum_volume_per_order'], errors='coerce')

# Consumo total global (para calcular participação)
consumo_total_global = df['median_article_consumption'].sum()

# Agregar por article_name (MP)
mp_resumo = df.groupby('article_name').agg(
    qtd_produtos=('product_name', 'nunique'),
    lista_produtos=('product_name', lambda x: ', '.join(sorted(x.unique()))),
    lista_status=('product_status', lambda x: ', '.join(sorted(x.unique()))),
    consumo_total=('median_article_consumption', 'sum'),
    article_unit=('article_unit', 'first'),
    minimum_volume_per_order=('minimum_volume_per_order', 'min'),
).reset_index()

# Garantir tipos numéricos (podem virar object após lambda aggs)
mp_resumo['consumo_total'] = pd.to_numeric(mp_resumo['consumo_total'], errors='coerce').astype(float)
mp_resumo['qtd_produtos'] = pd.to_numeric(mp_resumo['qtd_produtos'], errors='coerce')

# Participação no consumo total
mp_resumo['participacao_consumo_pct'] = (
    mp_resumo['consumo_total'] / consumo_total_global * 100
).round(2)

# Classificar a faixa de concentração
mp_resumo['faixa_concentracao'] = mp_resumo['qtd_produtos'].apply(
    lambda x: '1 produto (exclusiva)' if x == 1
    else '2 produtos (baixa flexibilidade)' if x == 2
    else '3+ produtos (compartilhada)'
)

# Ordenar: menor quantidade de produtos primeiro, depois maior consumo
mp_resumo = mp_resumo.sort_values(
    ['qtd_produtos', 'consumo_total'],
    ascending=[True, False]
).reset_index(drop=True)

print(f"Total de MPs/artigos distintos: {len(mp_resumo)}")
print()
print("Distribuição por faixa de concentração:")
print(mp_resumo['faixa_concentracao'].value_counts().sort_index())
print()
print(f"Consumo total global: {consumo_total_global:,.2f}")

Total de MPs/artigos distintos: 54

Distribuição por faixa de concentração:
faixa_concentracao
1 produto (exclusiva)               16
2 produtos (baixa flexibilidade)    10
3+ produtos (compartilhada)         28
Name: count, dtype: int64

Consumo total global: 495.54


In [44]:
# Tabela de diagnóstico: MPs mais críticas (1 ou 2 produtos)
diagnostico_mp_skp_concentracao = mp_resumo[
    mp_resumo['qtd_produtos'] <= 2
].copy()

print(f"MPs com concentração crítica (≤2 produtos): {len(diagnostico_mp_skp_concentracao)}")
print(f"  - Exclusivas (1 produto): {(diagnostico_mp_skp_concentracao['qtd_produtos'] == 1).sum()}")
print(f"  - Baixa flexibilidade (2 produtos): {(diagnostico_mp_skp_concentracao['qtd_produtos'] == 2).sum()}")
print()
diagnostico_mp_skp_concentracao[
    ['article_name', 'faixa_concentracao', 'qtd_produtos', 'lista_produtos',
     'lista_status', 'consumo_total', 'participacao_consumo_pct', 'article_unit']
]

MPs com concentração crítica (≤2 produtos): 26
  - Exclusivas (1 produto): 16
  - Baixa flexibilidade (2 produtos): 10



,article_name,faixa_concentracao,qtd_produtos,lista_produtos,lista_status,consumo_total,participacao_consumo_pct,article_unit
0,Grafiato,1 produto (exclusiva),1,Camiseta Manga Curta TrainIN Masculino,ativo_em_lancamento,241.000,48.63,kg
1,Mac Puelon,1 produto (exclusiva),1,Maxi Saia NYIN Feminino,ativo_perene,2.400,0.48,m
2,Haiti,1 produto (exclusiva),1,Parka Neutral,desativado,1.189,0.24,kg
3,Mac Power High Tech,1 produto (exclusiva),1,Bolsa Utility Transversal Feminino,desativado,1.000,0.20,kg
4,Kylie,1 produto (exclusiva),1,Calça Director Masculino,ativo_em_lancamento,0.740,0.15,kg
5,Ultra Slim,1 produto (exclusiva),1,Calça Director Masculino,ativo_em_lancamento,0.618,0.12,kg
6,Silk Span,1 produto (exclusiva),1,Calça Director Masculino,ativo_em_lancamento,0.480,0.10,m
7,MVS Thirty Plus,1 produto (exclusiva),1,Daily Light T-shirt Masculino,ativo_perene,0.300,0.06,kg
8,Outlast,1 produto (exclusiva),1,Performance T-shirt 2.0 Masculino,ativo_perene,0.165,0.03,kg
9,Fluity,1 produto (exclusiva),1,The Perfect Top Feminino,ativo_perene,0.118,0.02,kg


In [45]:
# Visão completa de todas as MPs (para referência)
mp_resumo[
    ['article_name', 'faixa_concentracao', 'qtd_produtos', 'lista_produtos',
     'consumo_total', 'participacao_consumo_pct']
]

,article_name,faixa_concentracao,qtd_produtos,lista_produtos,consumo_total,participacao_consumo_pct
0,Grafiato,1 produto (exclusiva),1,Camiseta Manga Curta TrainIN Masculino,241.0000,48.63
1,Mac Puelon,1 produto (exclusiva),1,Maxi Saia NYIN Feminino,2.4000,0.48
2,Haiti,1 produto (exclusiva),1,Parka Neutral,1.1890,0.24
3,Mac Power High Tech,1 produto (exclusiva),1,Bolsa Utility Transversal Feminino,1.0000,0.20
4,Kylie,1 produto (exclusiva),1,Calça Director Masculino,0.7400,0.15
5,Ultra Slim,1 produto (exclusiva),1,Calça Director Masculino,0.6180,0.12
6,Silk Span,1 produto (exclusiva),1,Calça Director Masculino,0.4800,0.10
7,MVS Thirty Plus,1 produto (exclusiva),1,Daily Light T-shirt Masculino,0.3000,0.06
8,Outlast,1 produto (exclusiva),1,Performance T-shirt 2.0 Masculino,0.1650,0.03
9,Fluity,1 produto (exclusiva),1,The Perfect Top Feminino,0.1180,0.02


## 7. Diagnóstico 2: Divergência de Cores dentro da mesma MP

Objetivo: para MPs compartilhadas (≥2 produtos), avaliar se as cores são de fato reaproveitadas entre os produtos.

Uma cor é **exclusiva** quando aparece em apenas um produto dentro de uma MP compartilhada.
Isso reduz a capacidade de reaproveitamento — a MP pode parecer compartilhada, mas na prática funciona como exclusiva para aquela cor.

### Interpretação do `color_status`

| `color_status` | Significado | Nível de risco/oportunidade |
|---|---|---|
| `ativo_perene` | Cor comprometida no portfólio | **Risco real** — prioridade alta |
| `ativo_em_lancamento` | Cor lançada, perenização possível | **Oportunidade** — fácil redirecionar para cor já existente no perene |
| `ativo_capsula` | Cor teste, sem compromisso de manter | **Baixo risco** — monitorar |
| `desativado` | Cor inativa, mas cadastro existe | **Observação** — reativação mais fácil que criação nova |

### Facilidade de ação
- **Fácil**: lançamento/cápsula/desativado → cor pode ser redirecionada, substituída ou reativada
- **Difícil**: perene → cor comprometida, mudança requer decisão de produto

> **Nota**: inclui todos os `color_status` (perene, lançamento, cápsula, desativado) para visão completa do cadastro.

In [46]:
# Filtrar apenas MPs compartilhadas (≥2 produtos) na base de cores
produtos_por_mp = df_cores.groupby('article_name')['product_name'].nunique()
mps_compartilhadas = produtos_por_mp[produtos_por_mp >= 2].index
df_cores_compartilhadas = df_cores[df_cores['article_name'].isin(mps_compartilhadas)].copy()

print(f"MPs compartilhadas (≥2 produtos) na base de cores: {len(mps_compartilhadas)}")
print(f"Registros para análise de cores: {len(df_cores_compartilhadas)}")

# Para cada MP, listar todos os produtos
mp_todos_produtos = df_cores_compartilhadas.groupby('article_name')['product_name'].apply(
    lambda x: set(x.unique())
).to_dict()

# Para cada MP × cor, contar quantos produtos possuem essa cor e consolidar color_status
mp_cor_produtos = df_cores_compartilhadas.groupby(
    ['article_name', 'color']
).agg(
    produtos_com_cor=('product_name', lambda x: set(x.unique())),
    color_status=('color_status', 'first'),  # já consolidado na query
).reset_index()

mp_cor_produtos['qtd_produtos_com_cor'] = mp_cor_produtos['produtos_com_cor'].apply(len)

# Mapear total de produtos por MP
mp_cor_produtos['qtd_total_produtos_mp'] = mp_cor_produtos['article_name'].map(
    lambda mp: len(mp_todos_produtos[mp])
)

# Identificar produtos que usam a MP mas NÃO possuem essa cor
mp_cor_produtos['produtos_sem_cor'] = mp_cor_produtos.apply(
    lambda row: mp_todos_produtos[row['article_name']] - row['produtos_com_cor'],
    axis=1
)

# Classificar: exclusiva vs compartilhada
mp_cor_produtos['tipo_cor'] = mp_cor_produtos['qtd_produtos_com_cor'].apply(
    lambda x: 'exclusiva' if x == 1 else 'compartilhada'
)

# Facilidade de ação baseada no color_status
FACILIDADE_MAP = {
    'ativo_perene': 'difícil',
    'ativo_em_lancamento': 'fácil',
    'ativo_capsula': 'fácil',
    'desativado': 'fácil (reativar)',
}
mp_cor_produtos['facilidade_acao'] = mp_cor_produtos['color_status'].map(FACILIDADE_MAP).fillna('desconhecido')

# Formatar conjuntos para exibição
mp_cor_produtos['produtos_com_cor_str'] = mp_cor_produtos['produtos_com_cor'].apply(
    lambda s: ', '.join(sorted(s))
)
mp_cor_produtos['produtos_sem_cor_str'] = mp_cor_produtos['produtos_sem_cor'].apply(
    lambda s: ', '.join(sorted(s)) if s else '—'
)

print()
print("Distribuição de cores por tipo:")
print(mp_cor_produtos['tipo_cor'].value_counts())
print()
print("Distribuição de color_status nas cores exclusivas:")
exclusivas_mask = mp_cor_produtos['tipo_cor'] == 'exclusiva'
print(mp_cor_produtos.loc[exclusivas_mask, 'color_status'].value_counts())

MPs compartilhadas (≥2 produtos) na base de cores: 38
Registros para análise de cores: 1431

Distribuição de cores por tipo:
tipo_cor
compartilhada    299
exclusiva         75
Name: count, dtype: int64

Distribuição de color_status nas cores exclusivas:
color_status
desativado             55
ativo_perene           13
ativo_capsula           6
ativo_em_lancamento     1
Name: count, dtype: int64


In [47]:
# Tabela de diagnóstico: cores exclusivas em MPs compartilhadas
# Ordenar: perene primeiro (maior urgência), depois por MP e cor
STATUS_ORDER = {'ativo_perene': 0, 'ativo_em_lancamento': 1, 'ativo_capsula': 2, 'desativado': 3}

diagnostico_mp_cor_exclusiva = mp_cor_produtos[
    mp_cor_produtos['tipo_cor'] == 'exclusiva'
].copy()

diagnostico_mp_cor_exclusiva['_status_order'] = diagnostico_mp_cor_exclusiva['color_status'].map(STATUS_ORDER).fillna(9)
diagnostico_mp_cor_exclusiva = diagnostico_mp_cor_exclusiva.sort_values(
    ['_status_order', 'article_name', 'color']
).drop(columns=['_status_order']).reset_index(drop=True)

# Oportunidade contextualizada por color_status
def gerar_oportunidade(row):
    cs = row['color_status']
    cor = row['color']
    sem_cor = row['produtos_sem_cor_str']
    n_prods = row['qtd_total_produtos_mp']

    if cs == 'ativo_perene':
        return f"⚠ RISCO REAL: cor perene '{cor}' exclusiva — avaliar expandir para: {sem_cor}"
    elif cs == 'ativo_em_lancamento':
        return f"💡 OPORTUNIDADE: cor em lançamento '{cor}' — avaliar perenizar em cor já existente no perene ou expandir"
    elif cs == 'ativo_capsula':
        return f"🔍 MONITORAR: cor cápsula '{cor}' — teste, sem ação imediata necessária"
    elif cs == 'desativado':
        return f"📋 CADASTRO: cor desativada '{cor}' — reativação possível se necessário (mais fácil que criar nova)"
    else:
        return f"Avaliar cor '{cor}' nos produtos: {sem_cor}"

diagnostico_mp_cor_exclusiva['oportunidade'] = diagnostico_mp_cor_exclusiva.apply(gerar_oportunidade, axis=1)

print(f"Total de cores exclusivas em MPs compartilhadas: {len(diagnostico_mp_cor_exclusiva)}")
print(f"MPs afetadas: {diagnostico_mp_cor_exclusiva['article_name'].nunique()}")
print()
print("Breakdown por color_status:")
print(diagnostico_mp_cor_exclusiva['color_status'].value_counts())
print()

diagnostico_mp_cor_exclusiva[
    ['article_name', 'color', 'color_status', 'facilidade_acao',
     'produtos_com_cor_str', 'produtos_sem_cor_str',
     'qtd_total_produtos_mp', 'oportunidade']
]

Total de cores exclusivas em MPs compartilhadas: 75
MPs afetadas: 18

Breakdown por color_status:
color_status
desativado             55
ativo_perene           13
ativo_capsula           6
ativo_em_lancamento     1
Name: count, dtype: int64



,article_name,color,color_status,facilidade_acao,produtos_com_cor_str,produtos_sem_cor_str,qtd_total_produtos_mp,oportunidade
0,Authentico,Preto,ativo_perene,difícil,Hybrid Jogger Masculino,Lighter Jogger Masculino,2,⚠ RISCO REAL: cor perene 'Preto' exclusiva — a...
1,Dry Fitness,Mineral Brown,ativo_perene,difícil,Future Shorts 200 Masculino,"Hybrid Jogger Masculino, Lighter Jogger Masculino",3,⚠ RISCO REAL: cor perene 'Mineral Brown' exclu...
2,Dry Fitness,Nori Green,ativo_perene,difícil,Future Shorts 200 Masculino,"Hybrid Jogger Masculino, Lighter Jogger Masculino",3,⚠ RISCO REAL: cor perene 'Nori Green' exclusiv...
3,Dry Fitness,Urban Brown,ativo_perene,difícil,Future Shorts 200 Masculino,"Hybrid Jogger Masculino, Lighter Jogger Masculino",3,⚠ RISCO REAL: cor perene 'Urban Brown' exclusi...
4,Emana,Musgo,ativo_perene,difícil,Spectrum Socks Mid 2.0,"SneakIN Socks, Spectrum Socks High 2.0, Spectr...",4,⚠ RISCO REAL: cor perene 'Musgo' exclusiva — a...
...,...,...,...,...,...,...,...,...
70,Rib Stretch,Meteorite,desativado,fácil (reativar),Lighter Jogger Masculino,Hybrid Jogger Masculino,2,📋 CADASTRO: cor desativada 'Meteorite' — reati...
71,Sportiva Pro,Off White,desativado,fácil (reativar),Energy Top Feminino,"Action Top Feminino, Easy Legging Feminino, Mo...",7,📋 CADASTRO: cor desativada 'Off White' — reati...
72,Sportiva Pro,Stone Gray,desativado,fácil (reativar),Motion Shorts Feminino,"Action Top Feminino, Easy Legging Feminino, En...",7,📋 CADASTRO: cor desativada 'Stone Gray' — reat...
73,Top Visco Comfort,Electric Tangerine,desativado,fácil (reativar),Turtleneck Feminino,"Structure Cropped Feminino, Structure Tank Fem...",5,📋 CADASTRO: cor desativada 'Electric Tangerine...


In [48]:
# Visão complementar: todas as cores por MP (compartilhadas + exclusivas)
mp_cor_produtos[
    ['article_name', 'color', 'color_status', 'facilidade_acao', 'tipo_cor',
     'qtd_produtos_com_cor', 'qtd_total_produtos_mp',
     'produtos_com_cor_str', 'produtos_sem_cor_str']
].sort_values(['article_name', 'tipo_cor', 'color'])

,article_name,color,color_status,facilidade_acao,tipo_cor,qtd_produtos_com_cor,qtd_total_produtos_mp,produtos_com_cor_str,produtos_sem_cor_str
0,Adapt Fit,Preto,ativo_perene,difícil,compartilhada,5,5,"Calça Flare InSkin Feminino, Modern Top Femini...",—
1,Authentico,Azzure,desativado,fácil (reativar),exclusiva,1,2,Lighter Jogger Masculino,Hybrid Jogger Masculino
2,Authentico,Meteorite,desativado,fácil (reativar),exclusiva,1,2,Lighter Jogger Masculino,Hybrid Jogger Masculino
3,Authentico,Preto,ativo_perene,difícil,exclusiva,1,2,Hybrid Jogger Masculino,Lighter Jogger Masculino
6,Bandagem,Concrete,desativado,fácil (reativar),compartilhada,2,3,"Heavy Hoodie Masculino, Heavy Hoodie WTNB Masc...",Heavy Hoodie Nomad II Masculino
...,...,...,...,...,...,...,...,...,...
370,Vis UP,Preto,desativado,fácil (reativar),compartilhada,4,4,"Daily T-shirt Coco Bambu Feminino, Daily T-shi...",—
371,Vis UP,Purple Cloud,ativo_capsula,fácil,compartilhada,2,4,"Daily T-shirt Feminino, Daily T-shirt Masculino","Daily T-shirt Coco Bambu Feminino, Daily T-shi..."
372,Vis UP,Rain Forest,desativado,fácil (reativar),compartilhada,2,4,"Daily T-shirt Feminino, Daily T-shirt Masculino","Daily T-shirt Coco Bambu Feminino, Daily T-shi..."
373,Vis UP,Steel,desativado,fácil (reativar),compartilhada,2,4,"Daily T-shirt Feminino, Daily T-shirt Masculino","Daily T-shirt Coco Bambu Feminino, Daily T-shi..."


## 8. Tabelas Finais

In [49]:
# ===== Tabela Final 1: diagnostico_mp_skp_concentracao =====
diagnostico_mp_skp_concentracao_final = diagnostico_mp_skp_concentracao[
    ['article_name', 'faixa_concentracao', 'qtd_produtos', 'lista_produtos',
     'lista_status', 'consumo_total', 'participacao_consumo_pct', 'article_unit',
     'minimum_volume_per_order']
].copy()

print("=" * 60)
print("TABELA: diagnostico_mp_skp_concentracao")
print("=" * 60)
print(f"Linhas: {len(diagnostico_mp_skp_concentracao_final)}")
diagnostico_mp_skp_concentracao_final

TABELA: diagnostico_mp_skp_concentracao
Linhas: 26


,article_name,faixa_concentracao,qtd_produtos,lista_produtos,lista_status,consumo_total,participacao_consumo_pct,article_unit,minimum_volume_per_order
0,Grafiato,1 produto (exclusiva),1,Camiseta Manga Curta TrainIN Masculino,ativo_em_lancamento,241.000,48.63,kg,<NA>
1,Mac Puelon,1 produto (exclusiva),1,Maxi Saia NYIN Feminino,ativo_perene,2.400,0.48,m,1000
2,Haiti,1 produto (exclusiva),1,Parka Neutral,desativado,1.189,0.24,kg,<NA>
3,Mac Power High Tech,1 produto (exclusiva),1,Bolsa Utility Transversal Feminino,desativado,1.000,0.20,kg,<NA>
4,Kylie,1 produto (exclusiva),1,Calça Director Masculino,ativo_em_lancamento,0.740,0.15,kg,<NA>
5,Ultra Slim,1 produto (exclusiva),1,Calça Director Masculino,ativo_em_lancamento,0.618,0.12,kg,86
6,Silk Span,1 produto (exclusiva),1,Calça Director Masculino,ativo_em_lancamento,0.480,0.10,m,<NA>
7,MVS Thirty Plus,1 produto (exclusiva),1,Daily Light T-shirt Masculino,ativo_perene,0.300,0.06,kg,<NA>
8,Outlast,1 produto (exclusiva),1,Performance T-shirt 2.0 Masculino,ativo_perene,0.165,0.03,kg,153
9,Fluity,1 produto (exclusiva),1,The Perfect Top Feminino,ativo_perene,0.118,0.02,kg,153


In [50]:
# ===== Tabela Final 2: diagnostico_mp_cor_exclusiva =====
diagnostico_mp_cor_exclusiva_final = diagnostico_mp_cor_exclusiva[
    ['article_name', 'color', 'color_status', 'facilidade_acao',
     'produtos_com_cor_str', 'produtos_sem_cor_str',
     'tipo_cor', 'qtd_total_produtos_mp', 'oportunidade']
].copy()

print("=" * 60)
print("TABELA: diagnostico_mp_cor_exclusiva")
print("=" * 60)
print(f"Linhas: {len(diagnostico_mp_cor_exclusiva_final)}")
print()
print("Breakdown por color_status:")
print(diagnostico_mp_cor_exclusiva_final['color_status'].value_counts())
print()
print("Breakdown por facilidade_acao:")
print(diagnostico_mp_cor_exclusiva_final['facilidade_acao'].value_counts())
diagnostico_mp_cor_exclusiva_final

TABELA: diagnostico_mp_cor_exclusiva
Linhas: 75

Breakdown por color_status:
color_status
desativado             55
ativo_perene           13
ativo_capsula           6
ativo_em_lancamento     1
Name: count, dtype: int64

Breakdown por facilidade_acao:
facilidade_acao
fácil (reativar)    55
difícil             13
fácil                7
Name: count, dtype: int64


,article_name,color,color_status,facilidade_acao,produtos_com_cor_str,produtos_sem_cor_str,tipo_cor,qtd_total_produtos_mp,oportunidade
0,Authentico,Preto,ativo_perene,difícil,Hybrid Jogger Masculino,Lighter Jogger Masculino,exclusiva,2,⚠ RISCO REAL: cor perene 'Preto' exclusiva — a...
1,Dry Fitness,Mineral Brown,ativo_perene,difícil,Future Shorts 200 Masculino,"Hybrid Jogger Masculino, Lighter Jogger Masculino",exclusiva,3,⚠ RISCO REAL: cor perene 'Mineral Brown' exclu...
2,Dry Fitness,Nori Green,ativo_perene,difícil,Future Shorts 200 Masculino,"Hybrid Jogger Masculino, Lighter Jogger Masculino",exclusiva,3,⚠ RISCO REAL: cor perene 'Nori Green' exclusiv...
3,Dry Fitness,Urban Brown,ativo_perene,difícil,Future Shorts 200 Masculino,"Hybrid Jogger Masculino, Lighter Jogger Masculino",exclusiva,3,⚠ RISCO REAL: cor perene 'Urban Brown' exclusi...
4,Emana,Musgo,ativo_perene,difícil,Spectrum Socks Mid 2.0,"SneakIN Socks, Spectrum Socks High 2.0, Spectr...",exclusiva,4,⚠ RISCO REAL: cor perene 'Musgo' exclusiva — a...
...,...,...,...,...,...,...,...,...,...
70,Rib Stretch,Meteorite,desativado,fácil (reativar),Lighter Jogger Masculino,Hybrid Jogger Masculino,exclusiva,2,📋 CADASTRO: cor desativada 'Meteorite' — reati...
71,Sportiva Pro,Off White,desativado,fácil (reativar),Energy Top Feminino,"Action Top Feminino, Easy Legging Feminino, Mo...",exclusiva,7,📋 CADASTRO: cor desativada 'Off White' — reati...
72,Sportiva Pro,Stone Gray,desativado,fácil (reativar),Motion Shorts Feminino,"Action Top Feminino, Easy Legging Feminino, En...",exclusiva,7,📋 CADASTRO: cor desativada 'Stone Gray' — reat...
73,Top Visco Comfort,Electric Tangerine,desativado,fácil (reativar),Turtleneck Feminino,"Structure Cropped Feminino, Structure Tank Fem...",exclusiva,5,📋 CADASTRO: cor desativada 'Electric Tangerine...


## 9. Sugestões Iniciais de Endereçamento

As sugestões abaixo são **diagnósticos e oportunidades**, não decisões finais.

In [51]:
# ===== Tabela Final 3: sugestoes_oportunidades =====
sugestoes = []

# Sugestões para MPs exclusivas (1 produto)
for _, row in diagnostico_mp_skp_concentracao[
    diagnostico_mp_skp_concentracao['qtd_produtos'] == 1
].iterrows():
    sugestoes.append({
        'tipo_problema': 'concentracao_mp_exclusiva',
        'article_name': row['article_name'],
        'produto_afetado': row['lista_produtos'],
        'cor_afetada': '—',
        'color_status': '—',
        'facilidade_acao': '—',
        'sugestao_1': 'Avaliar se faz sentido manter essa MP no portfólio',
        'sugestao_2': 'Avaliar se outro produto poderia usar essa MP',
        'sugestao_3': 'Avaliar se o produto justifica a existência exclusiva da MP',
        'consumo_total': row['consumo_total'],
    })

# Sugestões para MPs com 2 produtos
for _, row in diagnostico_mp_skp_concentracao[
    diagnostico_mp_skp_concentracao['qtd_produtos'] == 2
].iterrows():
    sugestoes.append({
        'tipo_problema': 'concentracao_mp_baixa_flex',
        'article_name': row['article_name'],
        'produto_afetado': row['lista_produtos'],
        'cor_afetada': '—',
        'color_status': '—',
        'facilidade_acao': '—',
        'sugestao_1': 'Avaliar expandir uso da MP para mais produtos',
        'sugestao_2': 'Monitorar: risco moderado de baixa flexibilidade',
        'sugestao_3': '—',
        'consumo_total': row['consumo_total'],
    })

# Sugestões para cores exclusivas — contextualizadas por color_status
for _, row in diagnostico_mp_cor_exclusiva.iterrows():
    cs = row['color_status']
    cor = row['color']
    sem_cor = row['produtos_sem_cor_str']

    if cs == 'ativo_perene':
        s1 = f"PRIORIDADE ALTA: avaliar expandir cor perene '{cor}' para: {sem_cor}"
        s2 = 'Avaliar substituir por cor já compartilhada entre os produtos'
        s3 = 'Se não expandível, aceitar como risco estrutural documentado'
    elif cs == 'ativo_em_lancamento':
        s1 = f"OPORTUNIDADE: avaliar perenizar cor '{cor}' em cor já existente no perene"
        s2 = f"Alternativa: expandir cor '{cor}' para outros produtos: {sem_cor}"
        s3 = 'Se não perenizar, avaliar descontinuar e redirecionar'
    elif cs == 'ativo_capsula':
        s1 = f"MONITORAR: cor cápsula '{cor}' — sem ação imediata"
        s2 = 'Se converter para perene, reavaliar concentração'
        s3 = 'Se descontinuar, risco se resolve automaticamente'
    else:  # desativado
        s1 = f"CADASTRO DISPONÍVEL: cor desativada '{cor}' — reativação possível"
        s2 = f"Se necessário reativar, expandir para: {sem_cor}"
        s3 = 'Sem risco atual — apenas registro de oportunidade futura'

    sugestoes.append({
        'tipo_problema': 'cor_exclusiva_em_mp_compartilhada',
        'article_name': row['article_name'],
        'produto_afetado': row['produtos_com_cor_str'],
        'cor_afetada': cor,
        'color_status': cs,
        'facilidade_acao': row['facilidade_acao'],
        'sugestao_1': s1,
        'sugestao_2': s2,
        'sugestao_3': s3,
        'consumo_total': np.nan,
    })

sugestoes_oportunidades = pd.DataFrame(sugestoes)

print("=" * 60)
print("TABELA: sugestoes_oportunidades")
print("=" * 60)
print(f"Linhas: {len(sugestoes_oportunidades)}")
print()
print("Distribuição por tipo de problema:")
print(sugestoes_oportunidades['tipo_problema'].value_counts())
print()
print("Distribuição por color_status (onde aplicável):")
print(sugestoes_oportunidades[sugestoes_oportunidades['color_status'] != '—']['color_status'].value_counts())
print()
sugestoes_oportunidades

TABELA: sugestoes_oportunidades
Linhas: 101

Distribuição por tipo de problema:
tipo_problema
cor_exclusiva_em_mp_compartilhada    75
concentracao_mp_exclusiva            16
concentracao_mp_baixa_flex           10
Name: count, dtype: int64

Distribuição por color_status (onde aplicável):
color_status
desativado             55
ativo_perene           13
ativo_capsula           6
ativo_em_lancamento     1
Name: count, dtype: int64



,tipo_problema,article_name,produto_afetado,cor_afetada,color_status,facilidade_acao,sugestao_1,sugestao_2,sugestao_3,consumo_total
0,concentracao_mp_exclusiva,Grafiato,Camiseta Manga Curta TrainIN Masculino,—,—,—,Avaliar se faz sentido manter essa MP no portf...,Avaliar se outro produto poderia usar essa MP,Avaliar se o produto justifica a existência ex...,241.000
1,concentracao_mp_exclusiva,Mac Puelon,Maxi Saia NYIN Feminino,—,—,—,Avaliar se faz sentido manter essa MP no portf...,Avaliar se outro produto poderia usar essa MP,Avaliar se o produto justifica a existência ex...,2.400
2,concentracao_mp_exclusiva,Haiti,Parka Neutral,—,—,—,Avaliar se faz sentido manter essa MP no portf...,Avaliar se outro produto poderia usar essa MP,Avaliar se o produto justifica a existência ex...,1.189
3,concentracao_mp_exclusiva,Mac Power High Tech,Bolsa Utility Transversal Feminino,—,—,—,Avaliar se faz sentido manter essa MP no portf...,Avaliar se outro produto poderia usar essa MP,Avaliar se o produto justifica a existência ex...,1.000
4,concentracao_mp_exclusiva,Kylie,Calça Director Masculino,—,—,—,Avaliar se faz sentido manter essa MP no portf...,Avaliar se outro produto poderia usar essa MP,Avaliar se o produto justifica a existência ex...,0.740
...,...,...,...,...,...,...,...,...,...,...
96,cor_exclusiva_em_mp_compartilhada,Rib Stretch,Lighter Jogger Masculino,Meteorite,desativado,fácil (reativar),CADASTRO DISPONÍVEL: cor desativada 'Meteorite...,"Se necessário reativar, expandir para: Hybrid ...",Sem risco atual — apenas registro de oportunid...,NaN
97,cor_exclusiva_em_mp_compartilhada,Sportiva Pro,Energy Top Feminino,Off White,desativado,fácil (reativar),CADASTRO DISPONÍVEL: cor desativada 'Off White...,"Se necessário reativar, expandir para: Action ...",Sem risco atual — apenas registro de oportunid...,NaN
98,cor_exclusiva_em_mp_compartilhada,Sportiva Pro,Motion Shorts Feminino,Stone Gray,desativado,fácil (reativar),CADASTRO DISPONÍVEL: cor desativada 'Stone Gra...,"Se necessário reativar, expandir para: Action ...",Sem risco atual — apenas registro de oportunid...,NaN
99,cor_exclusiva_em_mp_compartilhada,Top Visco Comfort,Turtleneck Feminino,Electric Tangerine,desativado,fácil (reativar),CADASTRO DISPONÍVEL: cor desativada 'Electric ...,"Se necessário reativar, expandir para: Structu...",Sem risco atual — apenas registro de oportunid...,NaN


## 10. Resumo Executivo

> O resumo abaixo é gerado automaticamente a partir dos dados processados.

In [52]:
# --- Métricas do Diagnóstico 1 ---
total_mps = len(mp_resumo)
mps_1_prod = (mp_resumo['qtd_produtos'] == 1).sum()
mps_2_prod = (mp_resumo['qtd_produtos'] == 2).sum()
mps_3plus = (mp_resumo['qtd_produtos'] >= 3).sum()

# Top 5 MPs exclusivas por consumo
top5_exclusivas = diagnostico_mp_skp_concentracao[
    diagnostico_mp_skp_concentracao['qtd_produtos'] == 1
].nlargest(5, 'consumo_total')[['article_name', 'lista_produtos', 'consumo_total']]

# --- Métricas do Diagnóstico 2 ---
total_cores_exclusivas = len(diagnostico_mp_cor_exclusiva)
mps_afetadas_cor = diagnostico_mp_cor_exclusiva['article_name'].nunique()

# Breakdown por color_status
cores_por_status = diagnostico_mp_cor_exclusiva['color_status'].value_counts()
cores_perene = cores_por_status.get('ativo_perene', 0)
cores_lancamento = cores_por_status.get('ativo_em_lancamento', 0)
cores_capsula = cores_por_status.get('ativo_capsula', 0)
cores_desativado = cores_por_status.get('desativado', 0)

# Top 5 MPs com mais cores exclusivas perenes
top5_mp_cores_perenes = diagnostico_mp_cor_exclusiva[
    diagnostico_mp_cor_exclusiva['color_status'] == 'ativo_perene'
].groupby('article_name').size().nlargest(5)

# --- Imprimir Resumo ---
resumo = f"""
{'='*60}
RESUMO EXECUTIVO — Diagnóstico MP ↔ SKP
{'='*60}

📊 VISÃO GERAL
  Total de MPs/artigos analisados: {total_mps}
  MPs exclusivas (1 produto):     {mps_1_prod} ({mps_1_prod/total_mps*100:.1f}%)
  MPs baixa flexibilidade (2):    {mps_2_prod} ({mps_2_prod/total_mps*100:.1f}%)
  MPs compartilhadas (3+):        {mps_3plus} ({mps_3plus/total_mps*100:.1f}%)

📌 DIAGNÓSTICO 1 — Concentração MP → SKP
  {mps_1_prod} MPs atendem apenas 1 produto (risco alto)
  {mps_2_prod} MPs atendem 2 produtos (risco moderado)
  Top 5 MPs exclusivas por consumo:
"""

for _, row in top5_exclusivas.iterrows():
    resumo += f"    - {row['article_name']} → {row['lista_produtos']} (consumo: {row['consumo_total']:.4f})\n"

resumo += f"""
🎨 DIAGNÓSTICO 2 — Divergência de Cores
  Cores exclusivas encontradas:         {total_cores_exclusivas}
  MPs afetadas:                         {mps_afetadas_cor}

  Breakdown por color_status:
    ⚠ Perene (risco real):              {cores_perene}
    💡 Lançamento (oportunidade):        {cores_lancamento}
    🔍 Cápsula (monitorar):             {cores_capsula}
    📋 Desativado (cadastro):           {cores_desativado}

  Top 5 MPs com mais cores exclusivas PERENES:
"""

for mp, count in top5_mp_cores_perenes.items():
    resumo += f"    - {mp}: {count} cores perenes exclusivas\n"

resumo += f"""
{'='*60}
⚠ Este é um diagnóstico exploratório. Nenhuma decisão de negócio
  deve ser tomada diretamente a partir destes dados sem validação
  com as áreas de produto, planejamento e supply chain.
{'='*60}
"""

print(resumo)


RESUMO EXECUTIVO — Diagnóstico MP ↔ SKP

📊 VISÃO GERAL
  Total de MPs/artigos analisados: 54
  MPs exclusivas (1 produto):     16 (29.6%)
  MPs baixa flexibilidade (2):    10 (18.5%)
  MPs compartilhadas (3+):        28 (51.9%)

📌 DIAGNÓSTICO 1 — Concentração MP → SKP
  16 MPs atendem apenas 1 produto (risco alto)
  10 MPs atendem 2 produtos (risco moderado)
  Top 5 MPs exclusivas por consumo:
    - Grafiato → Camiseta Manga Curta TrainIN Masculino (consumo: 241.0000)
    - Mac Puelon → Maxi Saia NYIN Feminino (consumo: 2.4000)
    - Haiti → Parka Neutral (consumo: 1.1890)
    - Mac Power High Tech → Bolsa Utility Transversal Feminino (consumo: 1.0000)
    - Kylie → Calça Director Masculino (consumo: 0.7400)

🎨 DIAGNÓSTICO 2 — Divergência de Cores
  Cores exclusivas encontradas:         75
  MPs afetadas:                         18

  Breakdown por color_status:
    ⚠ Perene (risco real):              13
    💡 Lançamento (oportunidade):        1
    🔍 Cápsula (monitorar):             

## 11. Limitações e Próximos Passos

### Limitações

- **Sem informação de demanda**: não é possível ponderar o risco de concentração pelo volume de vendas ou previsão de demanda.
- **Sem MOQ/lote mínimo contextual**: o campo `minimum_volume_per_order` existe, mas não foi utilizado como critério de decisão nesta versão.
- **Sem tabela de equivalência de cores**: cores com nomes diferentes são tratadas como cores distintas, mesmo que visualmente sejam equivalentes.
- **Consumo mediano como proxy**: o consumo utilizado é a mediana por produto × artigo (grão SKU), que pode não refletir o volume real demandado.
- **Sem corte por margem ou relevância comercial**: todos os produtos ativos são tratados com o mesmo peso.
- **Consolidação de `color_status`**: usa hierarquia de prioridade (perene > lançamento > cápsula > desativado). Se um produto+cor tem SKUs em estados diferentes, o "melhor" estado é usado. Isso pode mascarar casos em que parte do tamanho está desativada.

### Campos ausentes ou premissas

| Campo | Status | Impacto |
|-------|--------|---------|
| `color` (via `integrated.skus`) | ✅ Disponível | Usado no Diagnóstico 2 |
| `sku_state` por cor (consolidado) | ✅ Disponível | Usado para classificar risco e facilidade de ação |
| Demanda/venda mensal por cor | ❌ Ausente | Impossibilita ponderar risco por volume |
| Tabela de equivalência de cores | ❌ Ausente | Cores similares tratadas como distintas |
| Margem por SKP | ❌ Ausente | Não é possível priorizar por rentabilidade |

### Próximos Passos

1. **Priorizar cores perenes exclusivas** — são o risco real e estrutural. Validar com produto/supply chain.
2. **Cores em lançamento exclusivas** — avaliar com produto se perenização deveria ocorrer em cor já existente no perene (reaproveitando MP).
3. **Cores cápsula** — monitorar durante o ciclo; se converter para perene, reavaliar concentração.
4. **Cores desativadas** — manter como registro de oportunidade; reativação é mais fácil que criação nova.
5. Considerar enriquecer a análise com dados de venda por cor para priorizar as oportunidades.
6. Avaliar a criação de uma tabela de equivalência de cores para reduzir falsos positivos no Diagnóstico 2.

In [53]:
# Export para clipboard (descomentar conforme necessário)

diagnostico_mp_skp_concentracao_final.to_clipboard(index=False, sep='\t', float_format='%.2f')
# diagnostico_mp_cor_exclusiva_final.to_clipboard(index=False, sep='\t', float_format='%.2f')
# sugestoes_oportunidades.to_clipboard(index=False, sep='\t', float_format='%.2f')

## 12. Export: Google Sheets — Matriz Produto × Cores

Gera uma planilha no Google Sheets com uma aba por MP compartilhada (≥2 produtos).
Cada aba contém:
- Título com nome da MP
- Contexto (quantidade de produtos, sobreposição média)
- Matriz: linhas = cores, colunas = produtos
- Células marcadas com ✓ coloridas por `color_status`:
  - 🟢 Verde = `ativo_perene`
  - 🔵 Azul = `ativo_em_lancamento` ou `ativo_capsula`
  - 🔴 Vermelho = `desativado`
- Coluna "Cobertura" com % de produtos que possuem cada cor

In [54]:
import gspread
from gspread_formatting import (
    CellFormat, Color, TextFormat, format_cell_range,
    set_column_width, set_row_height, batch_updater
)
import time

from pathlib import Path
from google.oauth2.service_account import Credentials

# Autenticação
scope = [
    'https://www.googleapis.com/auth/spreadsheets',
    'https://www.googleapis.com/auth/drive',
]
SERVICE_ACCOUNT_FILE = Path('../secrets/google-service-account.json')
if not SERVICE_ACCOUNT_FILE.exists():
    raise FileNotFoundError(
        f"Salve a chave JSON da service account em: {SERVICE_ACCOUNT_FILE.resolve()}"
    )

creds = Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=scope)
gc = gspread.authorize(creds)

# Abrir planilha existente
sh = gc.open_by_url('https://docs.google.com/spreadsheets/d/1IPT94cthkZgJhvXjLhosG6FFnmitcx-VbgKun7NhFR4')
print(f"Planilha: {sh.url}")

# Cores por status
COLOR_MAP = {
    'ativo_perene': Color(0.2, 0.66, 0.33),
    'ativo_em_lancamento': Color(0.26, 0.52, 0.96),
    'ativo_capsula': Color(0.26, 0.52, 0.96),
    'desativado': Color(0.87, 0.28, 0.27),
}
BG_COLOR_MAP = {
    'ativo_perene': Color(0.85, 0.95, 0.85),
    'ativo_em_lancamento': Color(0.85, 0.91, 0.98),
    'ativo_capsula': Color(0.85, 0.91, 0.98),
    'desativado': Color(0.98, 0.85, 0.85),
}

CAT_BG_COLOR = Color(0.96, 0.93, 0.82)  # bege para cabeçalho de categoria

# MPs compartilhadas ordenadas por qtd de produtos (desc)
mps_para_export = sorted(
    mps_compartilhadas,
    key=lambda mp: -len(mp_todos_produtos.get(mp, set()))
)

# Mapear abas existentes
abas_existentes = {ws.title: ws for ws in sh.worksheets()}

for i, mp_name in enumerate(mps_para_export):
    # if mp_name != 'Nylon WR 50+':
    #     continue
    mp_data = df_cores_compartilhadas[df_cores_compartilhadas['article_name'] == mp_name]

    # --- Filtrar produtos 100% desativados (todas as cores = desativado) ---
    produto_status_check = mp_data.groupby('product_name')['color_status'].apply(
        lambda x: (x == 'desativado').all()
    )
    produtos_totalmente_desativados = set(produto_status_check[produto_status_check].index)
    mp_data = mp_data[~mp_data['product_name'].isin(produtos_totalmente_desativados)]

    if len(mp_data) == 0 or mp_data['product_name'].nunique() < 2:
        continue  # Pular MPs que ficaram com <2 produtos após filtro

    # --- Scoring de produtos (para ordenação interna por grupo) ---
    produto_scores = {}
    produto_cat = {}
    for prod in mp_data['product_name'].unique():
        prod_data = mp_data[mp_data['product_name'] == prod]
        n_perene = (prod_data['color_status'] == 'ativo_perene').sum()
        n_lancamento = (prod_data['color_status'] == 'ativo_em_lancamento').sum()
        n_capsula = (prod_data['color_status'] == 'ativo_capsula').sum()
        n_desativado = (prod_data['color_status'] == 'desativado').sum()
        produto_scores[prod] = (-n_perene, -n_lancamento, -n_capsula, -n_desativado, prod)
        produto_cat[prod] = prod_data['category_4'].iloc[0]

    # --- Agrupar produtos por category_4, ordenação interna mantida ---
    from collections import defaultdict
    cat_grupos = defaultdict(list)
    for prod in mp_data['product_name'].unique():
        cat = produto_cat[prod]
        cat_grupos[cat].append(prod)

    # Ordenar dentro de cada grupo
    for cat in cat_grupos:
        cat_grupos[cat] = sorted(cat_grupos[cat], key=lambda p: produto_scores[p])

    # Categorias ordenadas alfabeticamente
    categorias_ordenadas = sorted(cat_grupos.keys())

    # Lista final de produtos (agrupados por categoria)
    produtos = []
    cat_spans = []  # (start_col_idx, end_col_idx, cat_name)
    for cat in categorias_ordenadas:
        start = len(produtos)
        produtos.extend(cat_grupos[cat])
        end = len(produtos) - 1
        cat_spans.append((start, end, cat))

    # --- Ordenar CORES por volume de checks (desc), depois alfabético ---
    cor_check_count = mp_data.groupby('color')['product_name'].nunique()
    cores = sorted(
        mp_data['color'].unique(),
        key=lambda c: (-cor_check_count.get(c, 0), c)
    )

    n_produtos = len(produtos)
    n_cores = len(cores)

    # Nome da aba (max 100 chars)
    aba_name = mp_name[:100] if len(mp_name) <= 100 else mp_name[:97] + "..."

    try:
        # Reusar aba existente ou criar nova
        if aba_name in abas_existentes:
            ws = abas_existentes[aba_name]
            ws.clear()
            if ws.row_count < n_cores + 12:
                ws.resize(rows=n_cores + 12)
            if ws.col_count < n_produtos + 5:
                ws.resize(cols=n_produtos + 5)
            time.sleep(2)
        else:
            ws = sh.add_worksheet(title=aba_name, rows=n_cores + 12, cols=n_produtos + 5)
            abas_existentes[aba_name] = ws
            time.sleep(2)

        # --- Linha 1: Título ---
        ws.update_cell(1, 1, f"{mp_name} — Cobertura de Cores por Produto")
        format_cell_range(ws, 'A1', CellFormat(
            textFormat=TextFormat(bold=True, fontSize=12)
        ))

        # --- Linha 3: Contexto ---
        cobertura_por_cor = mp_data.groupby('color')['product_name'].nunique()
        sobreposicao_media = (cobertura_por_cor / n_produtos * 100).mean()
        n_filtrados = len(produtos_totalmente_desativados)
        ctx_filtro = f" ({n_filtrados} produto(s) 100% desativado(s) filtrado(s))." if n_filtrados else "."
        contexto = (
            f"{n_produtos} produtos compartilham esta MP. "
            f"Sobreposição média: {sobreposicao_media:.0f}%. "
            f"Legenda: ✓ Verde = Perene, ✓ Azul = Lançamento/Cápsula, ✓ Vermelho = Desativado"
            f"{ctx_filtro}"
        )
        ws.update_cell(3, 1, contexto)
        format_cell_range(ws, 'A3', CellFormat(
            textFormat=TextFormat(italic=True, fontSize=9)
        ))
        time.sleep(2)

        # --- Linha 5: Cabeçalhos de CATEGORIA (acima dos produtos) ---
        cat_row = 5
        cat_header_cells = ['', '']  # Cor e Cobertura ficam vazios nesta linha
        for cat_start, cat_end, cat_name in cat_spans:
            span_len = cat_end - cat_start + 1
            cat_header_cells.append(cat_name)
            cat_header_cells.extend([''] * (span_len - 1))
        ws.update(f'A{cat_row}', [cat_header_cells])

        # Merge + formatar as células de categoria
        for cat_start, cat_end, cat_name in cat_spans:
            col_start = cat_start + 3  # +3 pq col 1=Cor, 2=Cobertura, 3=primeiro produto
            col_end = cat_end + 3
            if col_start != col_end:
                start_a1 = gspread.utils.rowcol_to_a1(cat_row, col_start)
                end_a1 = gspread.utils.rowcol_to_a1(cat_row, col_end)
                ws.merge_cells(f'{start_a1}:{end_a1}')
            # Formatar
            start_a1 = gspread.utils.rowcol_to_a1(cat_row, col_start)
            end_a1 = gspread.utils.rowcol_to_a1(cat_row, col_end)
            format_cell_range(ws, f'{start_a1}:{end_a1}', CellFormat(
                textFormat=TextFormat(bold=True, fontSize=9),
                backgroundColor=CAT_BG_COLOR,
                horizontalAlignment='CENTER',
            ))
        time.sleep(2)

        # --- Linha 6: Cabeçalhos de PRODUTO ---
        header_row = 6
        headers = ['Cor', 'Cobertura'] + produtos
        ws.update(f'A{header_row}', [headers])
        header_range = f'A{header_row}:{gspread.utils.rowcol_to_a1(header_row, len(headers))}'
        format_cell_range(ws, header_range, CellFormat(
            textFormat=TextFormat(bold=True, fontSize=10),
            backgroundColor=Color(0.93, 0.93, 0.93),
        ))
        time.sleep(2)

        # --- Dados da matriz ---
        data_rows = []
        for cor in cores:
            cor_data = mp_data[mp_data['color'] == cor]
            produtos_com_cor = set(cor_data['product_name'].unique())
            cobertura_pct = f"{len(produtos_com_cor)}/{n_produtos} ({len(produtos_com_cor)/n_produtos*100:.0f}%)"
            row = [cor, cobertura_pct]
            for prod in produtos:
                row.append('✓' if prod in produtos_com_cor else '—')
            data_rows.append(row)

        data_start_row = header_row + 1
        ws.update(f'A{data_start_row}', data_rows)
        time.sleep(2)

        # --- Aplicar cores com batch formatting ---
        with batch_updater(ws.spreadsheet) as batch:
            for row_idx, cor in enumerate(cores):
                cor_data = mp_data[mp_data['color'] == cor]
                produtos_com_cor = set(cor_data['product_name'].unique())
                cell_row = data_start_row + row_idx

                for col_idx, prod in enumerate(produtos):
                    if prod in produtos_com_cor:
                        status_row = cor_data[cor_data['product_name'] == prod]
                        if not status_row.empty:
                            cs = status_row.iloc[0]['color_status']
                            col_letter = gspread.utils.rowcol_to_a1(cell_row, col_idx + 3)
                            text_color = COLOR_MAP.get(cs, Color(0, 0, 0))
                            bg_color = BG_COLOR_MAP.get(cs, Color(1, 1, 1))
                            batch.format_cell_range(
                                ws, col_letter,
                                CellFormat(
                                    textFormat=TextFormat(bold=True, foregroundColor=text_color),
                                    backgroundColor=bg_color,
                                    horizontalAlignment='CENTER',
                                )
                            )

        # Centralizar cobertura
        format_cell_range(
            ws,
            f'B{data_start_row}:B{data_start_row + n_cores}',
            CellFormat(horizontalAlignment='CENTER')
        )

        if (i + 1) % 5 == 0:
            print(f"  Processadas {i+1}/{len(mps_para_export)} abas...")

    except Exception as e:
        print(f"  ⚠ Erro na aba '{aba_name}': {e}")

    time.sleep(2)

# Limpar aba padrão "Sheet1" se existir
if 'Sheet1' in abas_existentes and len(sh.worksheets()) > 1:
    try:
        sh.del_worksheet(abas_existentes['Sheet1'])
    except Exception:
        time.sleep(2)
        pass

# --- Reordenar abas por qtd de produtos (desc) ---
print("\nReordenando abas...")
time.sleep(2)
all_ws = sh.worksheets()
ws_by_title = {ws.title: ws for ws in all_ws}

for target_index, mp_name in enumerate(mps_para_export):
    aba_name = mp_name[:100] if len(mp_name) <= 100 else mp_name[:97] + "..."
    if aba_name in ws_by_title:
        ws = ws_by_title[aba_name]
        if ws.index != target_index:
            ws.update_index(target_index)
            time.sleep(2)

print(f"\n✅ Planilha atualizada com abas ordenadas por nº de produtos desc!")
print(f"🔗 {sh.url}")

FileNotFoundError: Salve a chave JSON da service account em: /Users/insider/LA_Coding_Projects/secrets/google-service-account.json